# Module 00 — Setup & Verification

Before running any cells, get the environment started:

## 1. Start the Docker stack

From the project root directory (the folder containing `docker-compose.yml`):

```bash
docker-compose up -d
```

This starts three containers:

| Container | Service | URL |
|-----------|---------|-----|
| `es01` | Elasticsearch | http://localhost:9200 |
| `kibana01` | Kibana | http://localhost:5601 |
| `minio01` | MinIO (S3-compatible) | http://localhost:9002 (API) / http://localhost:9003 (console) |

**First-start takes 1–2 minutes.** Watch progress with:

```bash
docker-compose logs -f
```

**Credentials** (set in `.env`):
- Elasticsearch / Kibana: `elastic` / `training123`
- MinIO console: `minioadmin` / `minioadmin`

## 2. Set up a Python virtual environment (once only)

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r notebooks/requirements.txt
```

Then open the Command Palette (`Ctrl+Shift+P`) → **Python: Select Interpreter** → pick `.venv`.

## 3. Run the cells below

This notebook:
1. Verifies Elasticsearch and Kibana are reachable
2. Inspects the cluster — version, health, nodes
3. Loads Kibana sample datasets (eCommerce, Flights, Logs)
4. Registers the shared filesystem snapshot repository used throughout the course

**Run this module first.** All subsequent modules depend on the sample data and repository created here.

**Stopping the environment:**
```bash
docker-compose down       # stop containers, keep data volumes
docker-compose down -v    # stop containers AND delete all data
```

---

In [11]:
import sys
sys.path.insert(0, '/home/jovyan/work')
from helpers import *

es = get_client()

from helpers import *

es = get_client()

In [12]:
heading('Elasticsearch — root info')
pp(dict(es.info()), 'GET /')

──────────────────────────────────────────── Elasticsearch — root info ────────────────────────────────────────────

╭──────────────────────────── GET / ────────────────────────────╮
│ {                                                             │
│   "name": "es01",                                             │
│   "cluster_name": "snapshot-training",                        │
│   "cluster_uuid": "jlmQErGESBO_cp3x1Lz4QQ",                   │
│   "version": {                                                │
│     "number": "9.3.0",                                        │
│     "build_flavor": "default",                                │
│     "build_type": "docker",                                   │
│     "build_hash": "17b451d8979a29e31935fe1eb901310350b30e62", │
│     "build_date": "2026-01-29T10:05:46.708397977Z",           │
│     "build_snapshot": false,                                  │
│     "lucene_version": "10.3.2",                               │
│     "minimum_wire_compatibility_version": "8.19.0",           │
│     "minimum_index_compatibility_version": "8.0.0"            │
│   },                                                          │
│   "tagline": "You Know, for Search"                           │
│ }                                                             │
╰───────────────────────────────────────────────────────────────╯

In [3]:
heading('Cluster health')
health = es.cluster.health()
pp(dict(health), 'GET /_cluster/health')

status = health['status']
if status in ('green', 'yellow'):
    success(f'Cluster is {status} — good to go!')
else:
    warn(f'Cluster status is {status} — investigate before continuing.')

───────────────────────────────────────────────── Cluster health ──────────────────────────────────────────────────

╭───────────────── GET /_cluster/health ─────────────────╮
│ {                                                      │
│   "cluster_name": "snapshot-training",                 │
│   "status": "yellow",                                  │
│   "timed_out": false,                                  │
│   "number_of_nodes": 1,                                │
│   "number_of_data_nodes": 1,                           │
│   "active_primary_shards": 48,                         │
│   "active_shards": 48,                                 │
│   "relocating_shards": 0,                              │
│   "initializing_shards": 0,                            │
│   "unassigned_shards": 3,                              │
│   "unassigned_primary_shards": 0,                      │
│   "delayed_unassigned_shards": 0,                      │
│   "number_of_pending_tasks": 0,                        │
│   "number_of_in_flight_fetch": 0,                      │
│   "task_max_waiting_in_queue_millis": 0,               │
│   "active_shards_percent_as_number": 94.11764705882352 │
│ }                                                      │
╰────────────────────────────────────────────────────────╯

✓ Cluster is yellow — good to go!

In [4]:
heading('Node info')
nodes = es.nodes.info()
for node_id, node in nodes['nodes'].items():
    console.print(f"  Node: [bold]{node['name']}[/bold]  version={node['version']}  roles={node['roles']}")

──────────────────────────────────────────────────── Node info ────────────────────────────────────────────────────

Node: es01  version=9.3.0  roles=['data', 'data_cold', 'data_content', 'data_frozen', 'data_hot', 'data_warm', 
'ingest', 'master', 'ml', 'remote_cluster_client', 'transform']

## 2. Kibana Reachability

In [5]:
heading('Kibana status')
status_resp = kibana_get('/api/status')
overall = status_resp.get('status', {}).get('overall', {})
kibana_ver = status_resp.get('version', {}).get('number', 'unknown')
console.print(f"  Kibana version : [bold]{kibana_ver}[/bold]")
console.print(f"  Overall status : [bold]{overall.get('level', '?')}[/bold] — {overall.get('summary', '')}")

if overall.get('level') == 'available':
    success('Kibana is available.')
else:
    warn('Kibana may not be fully ready — wait a moment and re-run.')

────────────────────────────────────────────────── Kibana status ──────────────────────────────────────────────────

Kibana version : 9.3.0

Overall status : available — All services and plugins are available

✓ Kibana is available.

## 3. Load Kibana Sample Datasets

We load all three Kibana sample datasets. These give us realistic indices, data views, dashboards
and visualizations to work with throughout the training.

| Dataset | Index | Contents |
|---------|-------|----------|
| eCommerce | `kibana_sample_data_ecommerce` | Orders, customers, products |
| Flights | `kibana_sample_data_flights` | Flight routes, delays, carriers |
| Logs | `kibana_sample_data_logs` | Web server access logs |

In [13]:
heading('Loading Kibana sample datasets')

for dataset in ['ecommerce', 'flights', 'logs']:
    try:
        load_sample_data(dataset)
    except Exception as exc:
        warn(f'{dataset}: {exc} (may already be loaded)')

───────────────────────────────────────── Loading Kibana sample datasets ──────────────────────────────────────────

✓ Sample dataset 'ecommerce' loaded.

✓ Sample dataset 'flights' loaded.

✓ Sample dataset 'logs' loaded.

In [7]:
# Confirm the indices exist
heading('Sample data indices')
indices = es.cat.indices(index='kibana_sample_data_*', h='index,docs.count,store.size', format='json')
t = Table('Index', 'Docs', 'Size')
for row in indices:
    t.add_row(row['index'], row['docs.count'], row['store.size'])
console.print(t)

─────────────────────────────────────────────── Sample data indices ───────────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Index                                         ┃ Docs  ┃ Size  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ .ds-kibana_sample_data_logs-2026.04.06-000001 │ 14074 │ 8.2mb │
│ kibana_sample_data_ecommerce                  │ 4675  │ 4mb   │
│ kibana_sample_data_flights                    │ 13014 │ 5.6mb │
└───────────────────────────────────────────────┴───────┴───────┘

## 4. Register the Filesystem Snapshot Repository

All modules use a single shared filesystem repository backed by the `./snapshot-repo/` directory
on your host machine (mounted into the ES container at `/usr/share/elasticsearch/snapshots`).

This means you can open a terminal and `ls snapshot-repo/` to see the raw snapshot files on disk.

In [14]:
heading('Register filesystem repository')
register_fs_repo(es)

# Verify it
verification = es.snapshot.verify_repository(name=FS_REPO_NAME)
pp(dict(verification), f'POST /_snapshot/{FS_REPO_NAME}/_verify')

───────────────────────────────────────── Register filesystem repository ──────────────────────────────────────────

✓ Repository 'training-fs-repo' registered at /usr/share/elasticsearch/snapshots

╭─ POST /_snapshot/training-fs-repo/_verify ─╮
│ {                                          │
│   "nodes": {                               │
│     "S_FtdbGzTsOR3WqKn8wt3g": {            │
│       "name": "es01"                       │
│     }                                      │
│   }                                        │
│ }                                          │
╰────────────────────────────────────────────╯

In [9]:
# Get full repository info
repo_info = es.snapshot.get_repository(name=FS_REPO_NAME)
pp(dict(repo_info), f'GET /_snapshot/{FS_REPO_NAME}')

╭─────────── GET /_snapshot/training-fs-repo ────────────╮
│ {                                                      │
│   "training-fs-repo": {                                │
│     "type": "fs",                                      │
│     "uuid": "xLa7DVyJQ6-A_IsY55H2UQ",                  │
│     "settings": {                                      │
│       "compress": "true",                              │
│       "location": "/usr/share/elasticsearch/snapshots" │
│     }                                                  │
│   }                                                    │
│ }                                                      │
╰────────────────────────────────────────────────────────╯

## 5. Take a Baseline Snapshot

Take a full snapshot now — we'll use it as a known-good restore point throughout the course.

In [15]:
heading('Creating baseline snapshot')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'baseline')

es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='baseline',
    body={
        'indices': ['kibana_sample_data_*'],
        'include_global_state': True,
        'feature_states': ['kibana'],
        'metadata': {
            'description': 'Baseline snapshot for snapshot-training course',
            'created_by': 'Module 00',
        },
    },
    wait_for_completion=False,
)

snap = wait_for_snapshot(es, FS_REPO_NAME, 'baseline')
pp(snap, 'Baseline snapshot info')

─────────────────────────────────────────── Creating baseline snapshot ────────────────────────────────────────────

ℹ Snapshot state: IN_PROGRESS — waiting...

╭─────────────────────── Baseline snapshot info ───────────────────────╮
│ {                                                                    │
│   "snapshot": "baseline",                                            │
│   "uuid": "8VwSOIoPRMqaob7d-UgR-Q",                                  │
│   "repository": "training-fs-repo",                                  │
│   "version_id": 9060000,                                             │
│   "version": "9.3.0",                                                │
│   "indices": [                                                       │
│     ".kibana_security_session_1",                                    │
│     ".kibana_usage_counters_9.3.0_001",                              │
│     ".apm-agent-configuration",                                      │
│     ".apm-custom-link",                                              │
│     ".kibana_alerting_cases_9.3.0_001",                              │
│     ".kibana_security_solution_9.3.0_001",                           │
│     ".kibana_ingest_9.3.0_001",                                      │
│     ".kibana_search_solution_9.3.0_001",                             │
│     ".ds-kibana_sample_data_logs-2026.04.06-000001",                 │
│     ".kibana_task_manager_9.3.0_001",                                │
│     ".kibana_analytics_9.3.0_001",                                   │
│     "kibana_sample_data_flights",                                    │
│     ".kibana_9.3.0_001",                                             │
│     "kibana_sample_data_ecommerce",                                  │
│     ".kibana_locks-000001"                                           │
│   ],                                                                 │
│   "data_streams": [                                                  │
│     "kibana_sample_data_logs"                                        │
│   ],                                                                 │
│   "include_global_state": true,                                      │
│   "metadata": {                                                      │
│     "description": "Baseline snapshot for snapshot-training course", │
│     "created_by": "Module 00"                                        │
│   },                                                                 │
│   "state": "SUCCESS",                                                │
│   "start_time": "2026-04-06T17:59:23.264Z",                          │
│   "start_time_in_millis": 1775498363264,                             │
│   "end_time": "2026-04-06T17:59:24.066Z",                            │
│   "end_time_in_millis": 1775498364066,                               │
│   "duration_in_millis": 802,                                         │
│   "failures": [],                                                    │
│   "shards": {                                                        │
│     "total": 15,                                                     │
│     "failed": 0,                                                     │
│     "successful": 15                                                 │
│   },                                                                 │
│   "feature_states": [                                                │
│     {                                                                │
│       "feature_name": "kibana",                                      │
│       "indices": [                                                   │
│         ".kibana_ingest_9.3.0_001",                                  │
│         ".kibana_security_solution_9.3.0_001",                       │
│         ".kibana_analytics_9.3.0_001",                               │
│         ".apm-custom-link",                                          │
│         ".kibana_task_manager_9.3.0_001",                            │
│         ".kibana_9.3.0_001",                                         │
│         ".apm-agent-configuration",                     

## Summary

| What | Status |
|------|--------|
| Elasticsearch 9.3.x | ✓ reachable |
| Kibana 9.3.x | ✓ reachable |
| Sample datasets (eCommerce, Flights, Logs) | ✓ loaded |
| Filesystem repository `training-fs-repo` | ✓ registered |
| Baseline snapshot | ✓ created |

**Next:** [01_repository_types.ipynb](01_repository_types.ipynb)